# 📊 SaleSight - Proceso ETL y EDA de los datos

### Hecho por:
- Stiven Posada Casadiego Cód. 67001289
- Josué Ribero Duarte Cód. 67001295

## 1. Extracción de los datos (E) 📤

In [ ]:
# Importación de las librerías que se usarán
import glob, os, zipfile
import sqlite3
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
import warnings
from kaggle.api.kaggle_api_extended import KaggleApi

# Libreria para registrar eventos
import logging

### Se realizó la extracción de los datos con un script de Web Scraping para Kaggle

In [ ]:
def extraction():
    # 1. Configurar las rutas de extracción y destino
    dataset = "sinjoysaha/sales-analysis-dataset"
    # mountboy/online-store-customer-transactions-1m-rows
    destino = r'/Users/josue/Documents/Escritorio/Universidad/ingenieria_de_datos/corte_1/Proyecto/SaleSight/data/raw'

    # Si no existe la ruta de destino, la crea
    if not os.path.exists(destino):
        os.makedirs(destino)

    # 2. Autenticación
    try:
        api = KaggleApi()
        api.authenticate()
        print("Autenticación exitosa.")
    except Exception as e:
        print(f"Error de autenticación: {e}")

    # 3. Descargar el archivo ZIP
    print(f"Descargando dataset: {dataset}...")
    api.dataset_download_files(dataset, path=destino, unzip=False)

    # 4. Descomprimir de forma dinámica
    archivos_en_destino = os.listdir(destino)
    zips = [archivo for archivo in archivos_en_destino if archivo.endswith('.zip')]

    if zips:
        # Tomamos el primer zip encontrado
        archivo_real = os.path.join(destino, zips[0])
        with zipfile.ZipFile(archivo_real, 'r') as referencia_zip:
            referencia_zip.extractall(destino)
            print(f"Archivos extraídos con éxito: {referencia_zip.namelist()}")
        
        # 5. Eliminar el ZIP para limpiar
        os.remove(archivo_real)
        print("Archivo ZIP eliminado.")
    else:
        print("No se encontró ningún archivo ZIP. Verifica si la descarga falló.")

In [ ]:
df = extraction()

## 2. Transformación de los datos (T) 📈

### 2.1 Se realizó la transformación de datos con pandas

In [ ]:
def transform():
    # 1. Definir la ruta de los datos
    ruta = r'/Users/josue/Documents/Escritorio/Universidad/ingenieria_de_datos/corte_1/Proyecto/SaleSight/data/raw/Sales-Analysis-Dataset'

    # 2. Buscar los archivos dentro de la carpeta
    archivos = glob.glob(os.path.join(ruta, "*.csv"))

    # 3. Leer todos los dataframes
    listaDf = [pd.read_csv(archivo) for archivo in archivos]

    # 4. Unir todos los dataframes en un solo archivo
    df = pd.concat(listaDf, ignore_index=True)

    # Se limpian los datos
    df = df.dropna(how='all')

    # Convertir a tipos numéricos
    df['Quantity Ordered'] = pd.to_numeric(df['Quantity Ordered'], errors='coerce')
    df['Price Each'] = pd.to_numeric(df['Price Each'], errors='coerce')

    # Borrar filas que quedaron en NaN tras la conversión
    df = df.dropna(subset=['Quantity Ordered', 'Price Each'])

    # Crear la columna de ventas totales
    df['Sales'] = df['Quantity Ordered'] * df['Price Each']

    # Convertir a datetime
    df['Order Date'] = pd.to_datetime(df['Order Date'])

    # Extraer solo la fecha (sin la hora) para agrupar por día
    df['Date'] = df['Order Date'].dt.date

    # Revisar el dataframe
    df

    # Leer la información del dataframe y los 5 primeros registros
    df.info()
    df.head()

    return df

In [ ]:
# Se ejecuta la transformación
df = transform()


## 3. Carga de los datos (L) 📈

### Se realizó la carga de los datos a una base de datos local y un archivo CSV.

In [ ]:
def df_to_csv():
    # Se crea el archivo csv y se envía a la ruta deseada
    nombre_csv = "processed_data.csv"
    ruta = r'/Users/josue/Documents/Escritorio/Universidad/ingenieria_de_datos/corte_1/Proyecto/SaleSight/data/processed'

    ruta_final = os.path.join(ruta, nombre_csv)

    df.to_csv(ruta_final, index=False)

In [ ]:
# Se construye un csv a partir del dataframe
df_to_csv()

In [ ]:
def df_to_db():
    # Se crea la ruta de destino y la conexión a la DB
    ruta = r'/Users/josue/Documents/Escritorio/Universidad/ingenieria_de_datos/corte_1/Proyecto/SaleSight/data/processed'
    nombre_db = "ventas_procesadas.db"
    ruta_final = os.path.join(ruta, nombre_db)

    # Conexión con la db
    conexion = sqlite3.connect(ruta_final)

    try:
        # Enviar el df a una tabla de nombre "ventas"
        df.to_sql('ventas', conexion, if_exists='replace', index=False)
    finally:
        conexion.close()

In [ ]:
df_to_db()